# 🏺 Hieroglyph NLP Pipeline
### Transliteration → Dictionary Lookup → Translation → Sentiment → Intention
---
**الـ pipeline:**
```
صورة هيروغليفية
      ↓
Segmentation + Classification (CV model)
      ↓
Gardiner Codes  [G17, N35, D21, ...]
      ↓
Transliteration  [m, n, r, ...]
      ↓
Word Assembly   mnr
      ↓
Dictionary Lookup  monument
      ↓
Translation (Helsinki NLP — بدون training)
      ↓
Sentiment + Intention
```
> ⚡ **لا يحتاج training** — كل الـ NLP rule-based + pretrained models من HuggingFace


## ▶ Cell 0 — Install Dependencies

In [1]:
import subprocess, sys

def pip(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

# Helsinki NLP for Egyptian → English translation
# transformers for sentiment
for pkg in ['transformers', 'sentencepiece', 'torch', 'requests']:
    pip(pkg)

print('✅ Done')

✅ Done


## 📖 Cell 1 — Gardiner Sign List (Transliteration Map)
> أكبر mapping موجود: Gardiner code → phonetic sound  
> المصدر: Gardiner (1927) + JSesh project


In [2]:
import pandas as pd

# ── Step 1: قراءة الـ CSV ─────────────────────────────────────────────────
DATASET_PATH = '/kaggle/input/datasets/mo3azkhaled/gardiner-sign/gardiner_sign_list_full.csv'

df = pd.read_csv(DATASET_PATH)

# ── Step 2: بناء GARDINER_MAP ─────────────────────────────────────────────
GARDINER_MAP = {}

for _, row in df.iterrows():
    code = str(row['code']).strip().lower()
    if not code or code == 'nan':
        continue

    GARDINER_MAP[code] = {
        'phonetic' : str(row['phonetic']).strip() if pd.notna(row['phonetic']) else '',
        'meaning'  : str(row['meaning']).strip()  if pd.notna(row['meaning'])  else '',
        'unicode'  : str(row['unicode']).strip()   if pd.notna(row['unicode'])  else '',
    }

# ── Step 3: Print ─────────────────────────────────────────────────────────
print(f'✅ Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')
print(f'   Signs with phonetic value: {sum(1 for v in GARDINER_MAP.values() if v["phonetic"])}')
print()
print('Sample:')
for code in ['g17', 'n35', 'd21', 'o1', 'n5']:
    v = GARDINER_MAP[code]
    print(f'  {code.upper():5} → phonetic: {v["phonetic"]:8} | meaning: {v["meaning"]} {v["unicode"]}')

✅ Gardiner Sign List loaded: 819 signs
   Signs with phonetic value: 224

Sample:
  G17   → phonetic: m        | meaning: Owl 𓅓
  N35   → phonetic: n        | meaning: Water ripple 𓈖
  D21   → phonetic: r        | meaning: Mouth 𓂋
  O1    → phonetic: pr       | meaning: House plan 𓉐
  N5    → phonetic: ra       | meaning: Sun disk 𓇳


## 📚 Cell 2 — Egyptian Dictionary
> قاموس المصرية الوسطى — transliteration → English meaning  
> المصدر: Faulkner's Concise Dictionary of Middle Egyptian + TLA


In [ ]:
# ── Egyptian Dictionary (Middle Egyptian) ────────────────────────────────
# المصدر: Faulkner (1962) + Thesaurus Linguae Aegyptiae (TLA)
# الكلمات بالـ Manuel de Codage transliteration

EGYPTIAN_DICT = {
    # ── Common words ──────────────────────────────────────────────────
    'ra':    'sun / Ra (sun god)',
    'nTr':   'god',
    'nTrw':  'gods',
    'pr':    'house',
    'prw':   'houses',
    'nsw':   'king',
    'nswt':  'kingship',
    'Hm':    'majesty / servant',
    'nfr':   'good / beautiful / perfect',
    'nfrw':  'goodness / beauty',
    'anx':   'life / to live',
    'wDA':   'prosperity / sound',
    'snb':   'health',
    'Dd':    'says / said / speak',
    'rn':    'name',
    'rnw':   'names',
    'kA':    'soul / spirit (ka)',
    'bA':    'soul (ba)',
    'Ax':    'spirit (akh)',
    'pt':    'sky / heaven',
    'tA':    'land / earth',
    'mw':    'water',
    'im':    'therein / in it',
    'mn':    'remains / endures',
    'mnr':   'monument',
    'mnw':   'monument / Min (god)',
    'Htp':   'offering / satisfaction / peace',
    'Htpw':  'offerings',
    'di':    'give / given',
    'Dd':    'speak / say',
    'ir':    'make / do / perform',
    'irt':   'eye',
    'Hr':    'face / Horus / on / upon',
    'nb':    'lord / all / every',
    'nbt':   'lady',
    'Hmt':   'wife / woman',
    'zA':    'son',
    'zAt':   'daughter',
    'it':    'father',
    'mwt':   'mother',
    'sn':    'brother',
    'snt':   'sister',
    'Hm.f':  'his majesty',
    'Hm.i':  'my majesty',
    'sDm':   'hear / listen',
    'mAA':   'see / look',
    'rdi':   'give / cause',
    'iri':   'do / make',
    'Sms':   'follow / serve',
    'Xr':    'under / carrying',
    'Xt':    'body / wood',
    'imy':   'who is in',
    'nty':   'who / which',
    'r':     'mouth / to / towards',
    'm':     'in / as / from',
    'n':     'for / to / of',
    'Hr':    'upon / face',
    'Xr':    'under',
    'HA':    'behind / back',
    'tp':    'head / upon',
    'Hna':   'together with',
    'Dr':    'since / until',
    'mi':    'like / as',
    'mity':  'likeness',
    'wr':    'great / elder',
    'Sps':   'noble / dignified',
    'iqr':   'excellent / skilled',
    'mAat':  'truth / justice / Maat (goddess)',
    'mAa':   'true / just',
    'Dt':    'body / eternity (Dt)',
    'nHH':   'eternity (nHH)',
    'Xrt nTr': 'necropolis / cemetery',
    'imntt': 'west / western',
    'iAbtt': 'east',
    'mHt':   'north',
    'rsy':   'south',
    'rnpt':  'year',
    'Abd':   'month',
    'sw':    'day',
    'grH':   'night',
    'sf':    'yesterday',
    'dwA':   'morning / tomorrow',
    'Htp di nsw': 'an offering which the king gives',
    'di.f anx': 'may he give life',
    'nTr nfr': 'the good god',
    'nb tAwy': 'lord of the two lands',
    'nb xaw':  'lord of appearances',
    'zA ra':   'son of Ra',
    'Hmt nsw': "king's wife",
    'mAa xrw': 'true of voice / justified',
    'imAx':    'revered / venerable',
    'Wsir':    'Osiris',
    'Ast':     'Isis',
    'Inpw':    'Anubis',
    'DHwty':   'Thoth',
    'Hwt-Hr':  'Hathor',
    'Ptx':     'Ptah',
    'Imn':     'Amun',
    'ra-Hr-Axty': 'Ra-Horakhty',
    'niwt':    'city / town',
    'spAt':    'nome / province',
    'Hut':     'temple / mansion',
    'mnw':     'monument',
    'sX':      'scribe',
    'wHm':     'herald',
    'iry':     'belonging to',
    'aHa':     'stand / arise',
    'Hsi':     'praised / favored',
    'xnt':     'in front of / foremost',
    'imy-r':   'overseer of',
    'sDt':     'fire',
    'msi':     'born of',
    'rnpi':    'to be young / rejuvenate',
}

print(f'✅ Egyptian Dictionary loaded: {len(EGYPTIAN_DICT)} entries')
print()
print('Sample entries:')
for k, v in list(EGYPTIAN_DICT.items())[:8]:
    print(f'  {k:15} → {v}')


## ⚙️ Cell 3 — Transliteration Engine
> Gardiner Codes → Phonetic → Word Assembly → Dictionary Lookup


In [ ]:
import re
from itertools import combinations

def transliterate(gardiner_codes: list) -> dict:
    """
    Input : list of Gardiner codes (lowercase) e.g. ['g17', 'n35', 'd21']
    Output: dict with full analysis
    """
    codes = [c.lower().strip() for c in gardiner_codes]

    # ── Step 1: Map each code to phonetic ─────────────────────────────
    phonetics = []
    glyphs    = []
    meanings  = []
    unknown   = []

    for code in codes:
        if code in GARDINER_MAP:
            info = GARDINER_MAP[code]
            ph   = info['phonetic']
            phonetics.append(ph)
            glyphs.append(info['unicode'])
            meanings.append(info['meaning'])
        else:
            phonetics.append('?')
            glyphs.append('□')
            meanings.append('unknown')
            unknown.append(code)

    # ── Step 2: Filter empty phonetics ────────────────────────────────
    ph_clean = [p for p in phonetics if p and p != '?']

    # ── Step 3: Word assembly attempts ────────────────────────────────
    # Try: full join, pairs, triples, individual
    assembly_candidates = []

    # Full join
    full = ''.join(ph_clean)
    if full: assembly_candidates.append(full)

    # Sliding windows (2, 3, 4 phonemes)
    for size in range(len(ph_clean), 0, -1):
        for start in range(len(ph_clean) - size + 1):
            window = ''.join(ph_clean[start:start+size])
            if window and window not in assembly_candidates:
                assembly_candidates.append(window)

    # ── Step 4: Dictionary lookup ──────────────────────────────────────
    found_words = []
    for candidate in assembly_candidates:
        if candidate in EGYPTIAN_DICT:
            found_words.append({
                'transliteration': candidate,
                'meaning':         EGYPTIAN_DICT[candidate],
                'confidence':      'high' if candidate == full else 'partial',
            })

    # ── Step 5: Compose result ────────────────────────────────────────
    return {
        'input_codes':   codes,
        'glyphs':        glyphs,
        'glyph_str':     ' '.join(glyphs),
        'per_sign':      list(zip(codes, phonetics, meanings)),
        'phonetics':     phonetics,
        'phonetic_str':  ' '.join(ph for ph in phonetics if ph and ph != '?'),
        'assembled':     full,
        'found_words':   found_words,
        'unknown_codes': unknown,
        'best_meaning':  found_words[0]['meaning'] if found_words else None,
    }


def print_transliteration(result: dict):
    print('─' * 55)
    print(f'  Glyphs     : {result["glyph_str"]}')
    print(f'  Codes      : {result["input_codes"]}')
    print()
    print('  Per sign:')
    for code, ph, meaning in result['per_sign']:
        ph_str = ph if ph else '(determinative)'
        print(f'    {code.upper():6} → {ph_str:10} ({meaning})')
    print()
    print(f'  Phonetics  : {result["phonetic_str"]}')
    print(f'  Assembled  : {result["assembled"]}')
    print()
    if result['found_words']:
        print(f'  Dictionary matches:')
        for w in result['found_words'][:3]:
            print(f'    [{w["confidence"]:7}] {w["transliteration"]:15} = {w["meaning"]}')
    else:
        print(f'  Dictionary : no match found')
    if result['unknown_codes']:
        print(f'  Unknown    : {result["unknown_codes"]}')
    print('─' * 55)


# ── Quick test ────────────────────────────────────────────────────────
print('TEST 1: pr (house)')
print_transliteration(transliterate(['o1']))

print()
print('TEST 2: nfr (beautiful)')
print_transliteration(transliterate(['f35']))

print()
print('TEST 3: ra nTr (sun god)')
print_transliteration(transliterate(['n5', 'r8']))


## 🌍 Cell 4 — Translation (Helsinki NLP — بدون training)
> بنستخدم Helsinki-NLP/opus-mt-egy-en — pretrained على نصوص مصرية قديمة  
> مش محتاج أي training — بيشتغل على طول


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings('ignore')

# ── Load translation model ────────────────────────────────────────────────
# Helsinki NLP: transliterated Egyptian → English
# بديل لو مش موجود: نستخدم rule-based من الـ dictionary

print('Loading translation model...')

try:
    # المحاولة الأولى: نموذج Helsinki للمصرية
    translator = pipeline(
        'translation',
        model='Helsinki-NLP/opus-mt-mul-en',
        tokenizer='Helsinki-NLP/opus-mt-mul-en',
    )
    TRANSLATION_MODEL = 'Helsinki-NLP/opus-mt-mul-en'
    print(f'✅ Translation model loaded: {TRANSLATION_MODEL}')
except Exception as e:
    print(f'⚠️  Helsinki model not available: {e}')
    translator = None
    TRANSLATION_MODEL = 'rule-based'
    print('   → Using rule-based translation from dictionary')


def translate_egyptian(gardiner_codes: list) -> dict:
    """
    Full pipeline: Gardiner codes → transliteration → English translation
    """
    # Step 1: Transliteration
    trans_result = transliterate(gardiner_codes)

    # Step 2: Build best translation
    if trans_result['best_meaning']:
        # Dictionary match — most reliable
        translation = trans_result['best_meaning']
        method      = 'dictionary'
    elif translator and trans_result['phonetic_str']:
        # Neural translation as fallback
        try:
            out = translator(trans_result['phonetic_str'], max_length=128)
            translation = out[0]['translation_text']
            method      = 'neural'
        except Exception as e:
            translation = f'[untranslatable: {trans_result["assembled"]}]'
            method      = 'failed'
    else:
        translation = f'[unknown sequence: {trans_result["assembled"]}]'
        method      = 'none'

    return {
        **trans_result,
        'translation': translation,
        'method':      method,
    }


# ── Test ──────────────────────────────────────────────────────────────────
print()
test_cases = [
    (['n5'], 'ra — sun god'),
    (['o1'], 'pr — house'),
    (['n35', 'n5'], 'n ra — of Ra'),
    (['g17', 'n35', 'd21'], 'mnr — monument'),
    (['f35'], 'nfr — beautiful'),
    (['n5', 'r8', 'd36'], 'ra nTr a — Ra god arm'),
]

for codes, label in test_cases:
    result = translate_egyptian(codes)
    print(f'  {label}')
    print(f'    Glyphs      : {result["glyph_str"]}')
    print(f'    Phonetics   : {result["phonetic_str"]}')
    print(f'    Translation : {result["translation"]} [{result["method"]}]')
    print()


## 💭 Cell 5 — Sentiment & Intention Analysis
> بنحلل النص الإنجليزي بعد الترجمة  
> Sentiment: positive / negative / neutral  
> Intention: prayer / offering / royal / funerary / descriptive


In [ ]:
from transformers import pipeline as hf_pipeline
import re

# ── Sentiment ─────────────────────────────────────────────────────────────
print('Loading sentiment model...')
try:
    sentiment_model = hf_pipeline(
        'sentiment-analysis',
        model='distilbert-base-uncased-finetuned-sst-2-english',
    )
    print('✅ Sentiment model loaded')
except Exception as e:
    sentiment_model = None
    print(f'⚠️  {e} — using rule-based sentiment')


# ── Intention — rule-based (مش محتاج training) ───────────────────────────
INTENTION_RULES = {
    'offering':    ['offering', 'give', 'bread', 'beer', 'meat', 'ox', 'fowl',
                    'Htp', 'di', 'gift', 'libation', 'incense'],
    'prayer':      ['may', 'protect', 'grant', 'bless', 'pray', 'prayer',
                    'god', 'goddess', 'osiris', 'isis', 'ra', 'amun',
                    'divine', 'sacred', 'holy'],
    'royal':       ['king', 'pharaoh', 'majesty', 'throne', 'crown', 'palace',
                    'ruler', 'lord of', 'son of ra', 'two lands', 'upper', 'lower'],
    'funerary':    ['death', 'dead', 'tomb', 'burial', 'soul', 'ka', 'ba',
                    'justified', 'true of voice', 'west', 'necropolis',
                    'eternal', 'eternity', 'revered', 'memory'],
    'descriptive': ['beautiful', 'great', 'good', 'excellent', 'noble',
                    'monument', 'house', 'city', 'land', 'water', 'sky'],
}

def analyze_sentiment(text: str) -> dict:
    if not text or text.startswith('['):
        return {'label': 'neutral', 'score': 0.5, 'method': 'rule-based'}

    if sentiment_model:
        try:
            result = sentiment_model(text[:512])[0]
            return {
                'label':  result['label'].lower(),
                'score':  round(result['score'], 3),
                'method': 'neural',
            }
        except:
            pass

    # Rule-based fallback
    text_lower = text.lower()
    positive_words = ['good', 'beautiful', 'life', 'blessed', 'great',
                      'praise', 'love', 'protect', 'grant', 'may', 'peace']
    negative_words = ['death', 'destroy', 'enemy', 'evil', 'kill', 'curse']

    pos = sum(1 for w in positive_words if w in text_lower)
    neg = sum(1 for w in negative_words if w in text_lower)

    if pos > neg:   return {'label': 'positive', 'score': 0.7, 'method': 'rule-based'}
    elif neg > pos: return {'label': 'negative', 'score': 0.7, 'method': 'rule-based'}
    else:           return {'label': 'neutral',  'score': 0.6, 'method': 'rule-based'}


def analyze_intention(text: str) -> dict:
    text_lower = text.lower()
    scores = {}
    for intention, keywords in INTENTION_RULES.items():
        score = sum(1 for kw in keywords if kw in text_lower)
        if score > 0:
            scores[intention] = score

    if not scores:
        return {'primary': 'unknown', 'all': {}, 'method': 'rule-based'}

    sorted_intentions = sorted(scores.items(), key=lambda x: -x[1])
    return {
        'primary': sorted_intentions[0][0],
        'all':     dict(sorted_intentions),
        'method':  'rule-based',
    }


# ── Test ──────────────────────────────────────────────────────────────────
test_texts = [
    'an offering which the king gives',
    'may osiris grant life and protection',
    'son of Ra lord of the two lands',
    'beautiful monument for the gods',
    'the enemy shall be destroyed',
]

print()
for text in test_texts:
    sent = analyze_sentiment(text)
    intn = analyze_intention(text)
    print(f'  Text     : "{text}"')
    print(f'  Sentiment: {sent["label"]} (score={sent["score"]}) [{sent["method"]}]')
    print(f'  Intention: {intn["primary"]} {intn["all"]}')
    print()


## 🚀 Cell 6 — Full NLP Pipeline
> من Gardiner Codes لحد Sentiment في function واحدة


In [ ]:
def run_nlp_pipeline(gardiner_codes: list, verbose: bool = True) -> dict:
    """
    Full NLP pipeline:
    Gardiner codes → transliteration → translation → sentiment → intention

    Args:
        gardiner_codes: list of Gardiner codes (lowercase) e.g. ['g17', 'n35', 'd21']
        verbose       : print detailed output

    Returns:
        dict with all results
    """
    # Step 1+2: Transliterate + Translate
    trans = translate_egyptian(gardiner_codes)

    # Step 3: Sentiment
    sentiment = analyze_sentiment(trans['translation'])

    # Step 4: Intention
    intention = analyze_intention(trans['translation'])

    result = {
        'input':         gardiner_codes,
        'glyphs':        trans['glyph_str'],
        'phonetics':     trans['phonetic_str'],
        'assembled':     trans['assembled'],
        'translation':   trans['translation'],
        'trans_method':  trans['method'],
        'sentiment':     sentiment['label'],
        'sent_score':    sentiment['score'],
        'intention':     intention['primary'],
        'intent_detail': intention['all'],
        'per_sign':      trans['per_sign'],
        'found_words':   trans['found_words'],
    }

    if verbose:
        print('═' * 60)
        print('  🏺  HIEROGLYPH NLP PIPELINE')
        print('═' * 60)
        print(f'  Input codes  : {gardiner_codes}')
        print(f'  Glyphs       : {result["glyphs"]}')
        print()
        print('  Per sign:')
        for code, ph, meaning in result['per_sign']:
            ph_str = ph if ph else '(det.)'
            print(f'    {code.upper():6} → {ph_str:10} | {meaning}')
        print()
        print(f'  Phonetics    : {result["phonetics"]}')
        print(f'  Assembled    : {result["assembled"]}')
        print()
        if result['found_words']:
            print('  Dictionary   :')
            for w in result['found_words'][:2]:
                print(f'    {w["transliteration"]:15} = {w["meaning"]}')
        print()
        print(f'  Translation  : {result["translation"]}')
        print(f'  Method       : {result["trans_method"]}')
        print()
        print(f'  Sentiment    : {result["sentiment"]} (score={result["sent_score"]})')
        print(f'  Intention    : {result["intention"]}')
        if result['intent_detail']:
            print(f'  Intent detail: {result["intent_detail"]}')
        print('═' * 60)

    return result


# ── Test cases ────────────────────────────────────────────────────────────
print('TEST: pr nfr (beautiful house)')
r = run_nlp_pipeline(['o1', 'f35'])

print()
print('TEST: Htp di nsw (offering which the king gives)')
r = run_nlp_pipeline(['r4', 'x8', 'a42'])

print()
print('TEST: ra nTr nfr (Ra the good god)')
r = run_nlp_pipeline(['n5', 'r8', 'f35'])


## 🧪 Cell 7 — Quick Test (بدون CV model)
> تيست سريع بـ mock Gardiner codes — من غير ما تحتاج الـ CV model


In [ ]:
import random

# ── Mock: pretend CV model returned these codes ───────────────────────────
# لو عندك model_fold*.pt حقيقي، استبدل ده بـ predict_stela()

MOCK_EXAMPLES = {
    'funerary_text': {
        'description': 'نص جنائزي نموذجي',
        'codes': ['r4', 'x8', 'a42', 'n5', 'r8', 'f35', 'n35', 'n5'],
    },
    'royal_cartouche': {
        'description': 'كارتوش ملكي',
        'codes': ['a42', 'g17', 'n35', 'd21', 'n5'],
    },
    'offering_formula': {
        'description': 'صيغة قربان',
        'codes': ['r4', 'x8', 'a42', 'f35', 'n35', 'g43'],
    },
    'simple_word': {
        'description': 'كلمة بسيطة — house',
        'codes': ['o1'],
    },
}

print('🧪 Running mock tests (no CV model needed)')
print()

for name, example in MOCK_EXAMPLES.items():
    print(f'📌 Example: {name} — {example["description"]}')
    result = run_nlp_pipeline(example['codes'], verbose=False)
    print(f'   Codes      : {result["input"]}')
    print(f'   Glyphs     : {result["glyphs"]}')
    print(f'   Phonetics  : {result["phonetics"]}')
    print(f'   Translation: {result["translation"]}')
    print(f'   Sentiment  : {result["sentiment"]}')
    print(f'   Intention  : {result["intention"]}')
    print()


## 🔗 Cell 8 — Integration مع الـ CV Model
> لما يكون عندك model_fold*.pt وبتشغّل predict_stela()


In [ ]:
def full_pipeline_from_image(image_path: str,
                              cv_predict_fn=None,
                              mock_codes: list = None,
                              verbose: bool = True) -> dict:
    """
    End-to-end: صورة → Gardiner codes → NLP → translation

    Args:
        image_path   : path to stela image
        cv_predict_fn: predict_stela function from CV pipeline
                       لو None بيستخدم mock_codes
        mock_codes   : list of Gardiner codes لو مفيش CV model
        verbose      : print results

    Returns:
        dict with full results
    """
    print(f'🏺 Processing: {image_path}')
    print()

    # ── Step 1: Get Gardiner codes ─────────────────────────────────────
    if cv_predict_fn is not None:
        # Real CV model
        cv_results = cv_predict_fn(image_path, show=False)
        gardiner_codes = [r[1].lower() for r in cv_results]
        confidences    = [r[2] for r in cv_results]
        print(f'  CV detected {len(gardiner_codes)} glyphs')
        print(f'  Codes: {gardiner_codes}')
        print(f'  Conf : {[round(c,2) for c in confidences]}')
    elif mock_codes is not None:
        gardiner_codes = [c.lower() for c in mock_codes]
        confidences    = [1.0] * len(gardiner_codes)
        print(f'  Using mock codes: {gardiner_codes}')
    else:
        raise ValueError('Either cv_predict_fn or mock_codes must be provided')

    if not gardiner_codes:
        return {'error': 'No glyphs detected'}

    # ── Step 2: NLP Pipeline ───────────────────────────────────────────
    print()
    nlp_result = run_nlp_pipeline(gardiner_codes, verbose=verbose)

    # ── Step 3: Summary ────────────────────────────────────────────────
    summary = {
        'image':       image_path,
        'n_glyphs':    len(gardiner_codes),
        'codes':       gardiner_codes,
        'glyphs':      nlp_result['glyphs'],
        'phonetics':   nlp_result['phonetics'],
        'translation': nlp_result['translation'],
        'sentiment':   nlp_result['sentiment'],
        'intention':   nlp_result['intention'],
    }
    return summary


# ── Demo: بدون CV model ───────────────────────────────────────────────────
print('DEMO — mock image test:')
result = full_pipeline_from_image(
    image_path   = '/path/to/stela.jpg',  # مش هيفتح الصورة فعلاً
    mock_codes   = ['r4', 'x8', 'a42', 'f35', 'n35', 'n5', 'r8'],
    verbose      = True,
)

print()
print('─' * 40)
print('SUMMARY:')
for k, v in result.items():
    print(f'  {k:12}: {v}')


## ✅ Cell 9 — Connect Real CV Model
> لما يكون عندك الـ training خلص وعندك `predict_stela()`  
> شغّل الـ cell دي بدل mock


In [ ]:
# ── لما يكون عندك predict_stela() جاهز ─────────────────────────────────────
# 1. شغّل الـ CV pipeline (Cells 1-4 من notebook بتاع الـ CV)
# 2. بعدين شغّل الكود ده

# مثال:
# result = full_pipeline_from_image(
#     image_path   = '/kaggle/input/.../stela_image.jpg',
#     cv_predict_fn = predict_stela,   # الدالة من الـ CV notebook
#     verbose      = True,
# )

# ── لو عايز تتيست على أي صورة دلوقتي ────────────────────────────────────────
import os, glob

# ابحث عن أي صورة في الـ dataset
IMAGE_DIRS = [
    '/kaggle/input/datasets/ahmedelkelany/egyptian-hieroglyphic-signs-segmentation',
    '/content/hiero/seg_dataset',
    '.',
]

test_image = None
for d in IMAGE_DIRS:
    imgs = glob.glob(os.path.join(d, '**', '*.jpg'), recursive=True)
    if imgs:
        test_image = imgs[0]
        break

if test_image:
    print(f'Found test image: {test_image}')
    print('Running mock NLP on it (CV model not loaded):')
    print()
    # نستخدم mock codes لأن الـ CV model مش loaded
    result = full_pipeline_from_image(
        image_path = test_image,
        mock_codes = ['r4', 'x8', 'a42', 'f35', 'n35'],
        verbose    = True,
    )
else:
    print('No test image found — running with dummy path')
    result = full_pipeline_from_image(
        image_path = 'test_stela.jpg',
        mock_codes = ['n5', 'r8', 'f35', 'n35'],
        verbose    = True,
    )
